# Config

In [15]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [16]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [30]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Interdisciplinario"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Interdisciplinario"]!="INDEFINIDO"]

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Interdisciplinario"])
savepath = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 193
Fold 0 - Val size: 257
Archivo guardado exitosamente en /tmp/final_project/dataSplits/interdiciplinario/train_test_ids_3folds.json


# 2) Entrenamiento

### Carga y entrenamiento

In [57]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [58]:
#Ruta de lectura
path2 = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path2, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
#filepath=os.path.join(path2, "data_translated_concat.csv")
#df = pd.read_csv(filepath)


In [60]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdisciplinario")

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdisciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
y_train = [1 if x == "SI" else 0 for x in y_train]
y_test = [1 if x == "SI" else 0 for x in y_test]

#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

(771, 16882) (193, 16882)


In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
#split_idx_path = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
split_idx_path = os.path.join(path2, "train_test_ids_3folds.json")
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

In [62]:
from utils.mlflow import eval_model

# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6839378238341969, 'f1_macro': 0.680506933702407, 'cm': array([[56, 26],
       [35, 76]]), 'precision': 0.7450980392156863, 'recall': 0.6846846846846847, 'f1_es': 0.6612903225806451, 'f1_en': 0.7145838845283241, 'cm_es': array([[22, 21],
       [21, 60]]), 'cm_en': array([[34,  5],
       [14, 16]])}
RandomForestClassifier
{'accuracy': 0.6683937823834197, 'f1_macro': 0.6474885844748859, 'cm': array([[41, 41],
       [23, 88]]), 'precision': 0.6821705426356589, 'recall': 0.7927927927927928, 'f1_es': 0.6285810963230317, 'f1_en': 0.679647676161919, 'cm_es': array([[12, 31],
       [11, 70]]), 'cm_en': array([[29, 10],
       [12, 18]])}
XGBClassifier
{'accuracy': 0.6476683937823834, 'f1_macro': 0.6274131274131274, 'cm': array([[40, 42],
       [26, 85]]), 'precision': 0.6692913385826772, 'recall': 0.7657657657657657, 'f1_es': 0.592274100830284, 'f1_en': 0.6949992071117412, 'cm_es': array([[11, 32],
       [15, 66]]), 'cm_en': array([[29, 10],
       [11, 

### Guardado de resultados

In [ ]:
import git
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import tempfile
import matplotlib.pyplot as plt
import json
import io
from PIL import Image

def metrics_lang(y, preds, lang_es):
    #Conversión en array
    y = np.array(y)
    preds = np.array(preds)
    lang_es = np.array(lang_es)

    # Seleccionar por máscara booleana
    y_es = y[lang_es] #Data originalmente en español
    preds_es = preds[lang_es]

    y_en = y[~lang_es] #Data originalmente en inglés
    preds_en = preds[~lang_es]
    
    #Computo de métricas
    f1_es = f1_score(y_es, preds_es, average="weighted")
    f1_en = f1_score(y_en, preds_en, average="weighted")
    cm_es = confusion_matrix(y_es, preds_es)
    cm_en = confusion_matrix(y_en, preds_en)

    return f1_es, f1_en, cm_es, cm_en

def eval_model(best_model, X_test, y_test, lang_es=None, mode = "binary"):
    results = {}
    preds = best_model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    #Métricas generales
    results = {
        'accuracy': accuracy_score(y_test, preds),
        'f1_macro': f1_score(y_test, preds, zero_division=0, average="macro"),
        'cm': cm,
    }
    # Métricas adicionales para clasificación binaria
    if mode == "binary":
        results.update({
            'precision': precision_score(y_test, preds, zero_division=0),
                'recall': recall_score(y_test, preds, zero_division=0)
            })
    # Métricas por idioma
    if lang_es is not None:
        f1_es, f1_en, cm_es, cm_en = metrics_lang(y_test, preds, lang_es)
        results.update({
            'f1_es': f1_es,
            'f1_en': f1_en,
            'cm_es': cm_es,
            'cm_en': cm_en
        })
    return results, preds

def register_confusion_matrix(df_cm, class_labels=None):
    """
    Genera una imagen de la matriz de confusión normalizada.

    Parámetros:
        df_cm (pd.DataFrame): matriz de confusión con índices y columnas como clases
        class_labels (list, opcional): etiquetas de clase a mostrar. 
                                       Si None, usa las del DataFrame.

    Retorna:
        PIL.Image con la matriz de confusión
    """
    cm = df_cm.to_numpy().astype(float)
    # Normalizar por filas (cada fila suma 1)
    cm = cm / cm.sum(axis=1, keepdims=True)

    # Etiquetas de clases
    if class_labels is None:
        class_labels = df_cm.columns.tolist()

    # Crear la figura
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues, vmin=0, vmax=1)
    ax.set_title("Matriz de Confusión")
    fig.colorbar(im, ax=ax)

    # Agregar valores dentro de cada celda
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:.3f}",
                    ha="center", va="center", color="black")

    # Configurar ticks con etiquetas correctas
    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels)
    ax.set_yticklabels(class_labels)

    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")

    plt.tight_layout()

    # Guardar en memoria como PNG
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)

    img = Image.open(buf)
    plt.close(fig)

    return img


def mlflow_ckeckpoint(path, results_val, models_dicc, X_test, y_test, df_test, save_preds=None, lang_es=None, mode_classification="binary"):
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    #Diccionario donde se guardará todo
    save_dict = {}

    #Make metrics folders
    path_metrics = os.path.join(path, "models")
    os.makedirs(save_path, exist_ok=True)

    #Make models folders
    path_models = os.path.join(path, "metrics")
    os.makedirs(save_path, exist_ok=True)

    for model_name, metrics in results_val.items():
       
        model = models_dicc[model_name]
        print(f"📝 Registrando modelo: {model_name}")

        # Hiperparámetros
        try:
            params = model.get_params()
            save_path = os.path.join(path_models, f"{model_name}.json")
            # Guardar en JSON
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(params, f, indent=4, ensure_ascii=False)

        except:
            print(f"⚠️ No se pudieron loggear los hiperparámetros para {model_name}")

        # Métricas de validación
        for k, v in metrics.items():
            save_dict[f"val_{k}"] = v

        #Predicciones y resultados de test
        results_test, preds = eval_model(model, X_test, y_test, lang_es, mode = mode_classification)

        #Guardar predicciones
        if save_preds is not None:
            df_test["y_true"]=y_test
            df_test["preds"]=preds
            df_test = df_test[["Código VRID", "y_true", "preds"]]
            #Save as csv
            save_path = os.path.join(path_metrics, f"preds_{model_name}.csv")
            df_test.to_csv(save_path, index=False, encoding="utf-8-sig")

        # Guardar métricas de test
        for k, v in results_test.items():
            if k.startswith("cm"):
                # Guardar confusion matrix (o similar) como artefacto
                # Guardar como CSV temporal
                v = pd.DataFrame(v)
                save_path = os.path.join(path_metrics, f"cm_{model_name}.csv")
                v.to_csv(save_path, index=False, encoding="utf-8-sig")
                #Guardar imagen
                cm_img = register_confusion_matrix(v)
                save_path = os.path.join(path_metrics, f"cm_{model_name}.png")
                cm_img.save(save_path, format="PNG")
                
            else:
                # Guardar métrica numérica
                save_dict[f"test_{k}"]  = v
        
        #Guardar commit de git
        save_dict["git_commit"] = commit_hash

        #Guardar diccionario con métricas
        save_path = os.path.join(path_metrics, f"{model_name}.json")
        # Guardar en JSON
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(params, f, indent=4, ensure_ascii=False)
